## ZFOURGE EAZY Notebook

A self-contained eazy-py notebook for computing photometric redshifts and
stellar population properties for the ZFOURGE survey fields (CDFS, COSMOS, UDS).
Catalogue data is downloaded automatically from Vizier; no local input files required.

Based on the eazy-py demo notebook by Gabriel Brammer.
eazy-py: https://github.com/gbrammer/eazy-py

### Operation
- `eazy_zfourge.ipynb` full pipeline from catalogue download to visualisation

### Field selection
Set `field = 'cdfs'`, `'cosmos'`, or `'uds'` in Cell 3 to switch between fields.

### Key steps
1. Environment setup and dependency installation
2. Catalogue download from Vizier (Straatman et al. 2016, J/ApJ/830/51)
3. Iterative zeropoint corrections
4. Photometric redshift fitting
5. Derived parameters: redshifts, rest-frame colours, stellar masses, SFRs
6. Interactive visualisation via Plotly/Dash

### Output files
- `zfourge.{field}.zspec.vizier.csv` input catalogue
- `zfourge.{field}.zout.fits` redshifts and stellar population parameters
- `zfourge.{field}.zphot.zeropoint` derived zeropoint corrections
- `zfourge.{field}.h5` full fit data for SED plotting
- `zfourge.{field}.output.fits` exported catalogue for further analysis

### References
Straatman et al. (2016), ApJ, 830, 51

### Running this notebook
The recommended option for new users is Google Colab as no local installation required.
Open the notebook in Colab, run Cell 1, and all dependencies will be installed automatically.

For local use, ensure you have Python 3.10 or later and Jupyter installed before opening.

In [ ]:
# ── Environment setup ────────────────────────────────

import os
import sys

def pip_install(package):
    os.system(f'{sys.executable} -m pip install {package} -q')

try:
    import eazy
except ImportError:
    pip_install('eazy')

try:
    import grizli
except ImportError:
    pip_install('grizli')

try:
    import dust_attenuation
except ImportError:
    pip_install('git+https://github.com/karllark/dust_attenuation.git')

try:
    import dash
except ImportError:
    pip_install('"eazy[vistool]"')

try:
    import pandas
except ImportError:
    pip_install('pandas')

try:
    import astroquery
except ImportError:
    pip_install('astroquery')

try:
    import ipywidgets
except ImportError:
    pip_install('ipywidgets')

import eazy.utils
import eazy.photoz
if not os.path.exists(eazy.utils.DATA_PATH):
    eazy.fetch_eazy_photoz()

print('Setup complete.')

In [ ]:
# ── Imports and version check ───────────────────────
import os
import sys
import time
import importlib
import warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import eazy
import eazy.utils
import eazy.photoz
import eazy.hdf5
from astropy.utils.exceptions import AstropyWarning

np.seterr(all='ignore')
warnings.simplefilter('ignore', category=AstropyWarning)

print('eazy.utils.DATA_PATH = ' + eazy.utils.DATA_PATH + '\n')
print(time.ctime() + '\n')
print(sys.version + '\n')

for module in ['numpy', 'scipy', 'matplotlib', 'astropy', 'eazy']:
    mod = importlib.import_module(module)
    print('{0:>20} : {1}'.format(module, mod.__version__))

In [ ]:
# ── Field selector and catalogue download ────────────
import ipywidgets as widgets
from IPython.display import display

field_widget = widgets.ToggleButtons(
    options=['cdfs', 'cosmos', 'uds'],
    description='Field:',
    button_style='info'
)
display(field_widget)

In [ ]:
# ── Download catalogue ──────────────────────────────
from grizli.catalog import query_tap_catalog

field = field_widget.value

radec = {
    'cdfs':   (53.0,  -27.9),
    'cosmos': (150.1,   2.2),
    'uds':    (34.3,   -5.3),
}

MAX_KMAG = 22.5
db = f'"J/ApJ/830/51/zf_{field}"'
cat_name = f'zfourge.{field}.zspec.vizier.csv'

if not os.path.exists(cat_name):
    zf = query_tap_catalog(
    *radec[field],       # unpack (ra, dec) as separate arguments
    radius=60,           # cone search radius in arcminutes
    db=db,               # Vizier table name
    vizier=True,         # query via Vizier TAP service
    extra=f" AND (zspec > 0 OR FKsall > {10**(-0.4*(MAX_KMAG-25))})",  # keep spec-z sources or brighter than MAX_KMAG
    verbose=True         # print download progress
    )
    print(f'\nFound {len(zf)} objects in {db}')
    zf.write(cat_name, overwrite=True)
else:
    import grizli.utils
    zf = grizli.utils.read_catalog(cat_name)

print('Catalogue: ', cat_name)

In [ ]:
# ── Cell 4: Parameters and translate file ────────────────────
trans_name = f'zfourge.{field}.vizier.translate.csv'

if os.path.exists(trans_name):
    print(f'Using local translate file: {trans_name}')
else:
    raise FileNotFoundError(f'Translate file {trans_name} not found')

params = {}
params['CATALOG_FILE'] = cat_name               # input catalogue
params['CATALOG_FORMAT'] = 'csv'                # catalogue format
params['MAIN_OUTPUT_FILE'] = f'zfourge.{field}' # output file prefix
params['MW_EBV'] = eazy.utils.get_irsa_dust(
    np.nanmedian(zf['ra']), np.nanmedian(zf['dec'])
)                                               # Milky Way dust extinction from IRSA
params['CAT_HAS_EXTCORR'] = True                # catalogue fluxes are already extinction corrected
params['PRIOR_ABZP'] = 25                       # AB zeropoint for prior magnitude
params['PRIOR_FILTER'] = 255                    # Ks filter used for magnitude prior
params['PRIOR_FILE'] = 'templates/prior_K_TAO.dat'  # redshift prior file
params['TEMPLATES_FILE'] = 'templates/spline_templates_v2/tweak_spline.param'   # SED template set
params['SYS_ERR'] = 0.03                        # systematic flux error floor (3%)
params['Z_MIN'] = 0.01                          # minimum redshift
params['Z_MAX'] = 12.                           # maximum redshift
params['Z_STEP'] = 0.005                        # redshift grid step size
params['IGM_SCALE_TAU'] = 1.0                   # IGM opacity scaling
params['FIX_ZSPEC'] = False                     # do not fix redshifts to spec-z values

translate_file = trans_name

In [ ]:
# ── Initialise PhotoZ object ────────────────────────
self = eazy.photoz.PhotoZ(
    param_file=None,
    translate_file=translate_file,
    zeropoint_file=None,
    params=params,
    load_prior=True,
    load_products=False
)

In [ ]:
# ── Iterative zeropoint corrections ──────────────────
NITER = 3
NBIN = np.minimum(self.NOBJ // 100, 180)

self.param.params['VERBOSITY'] = 1.
for iter_ in range(NITER):
    print('Iteration: ', iter_)
    sn = self.fnu / self.efnu
    clip = (sn > 1).sum(axis=1) > 4
    self.iterate_zp_templates(
        idx=self.idx[clip],
        update_templates=False,
        update_zeropoints=True,
        iter=iter_,
        n_proc=4,
        save_templates=False,
        error_residuals=False,
        NBIN=NBIN,
        get_spatial_offset=False
    )

In [ ]:
# ── Fit catalogue and plot zphot vs zspec ────────────
self.set_sys_err(positive=True)
sample = np.isfinite(self.ZSPEC)
self.fit_catalog(self.idx[sample], n_proc=4)
fig = self.zphot_zspec()

In [ ]:
# ── Derived parameters ───────────────────────────────
warnings.simplefilter('ignore', category=RuntimeWarning)

zout, hdu = self.standard_output(
    simple=False,
    rf_pad_width=0.5,
    rf_max_err=2,
    prior=True,
    beta_prior=True,
    absmag_filters=[],
    extra_rf_filters=[]
)

print(zout.colnames)

In [ ]:
# ── UVJ diagram coloured by sSFR ────────────────────
uv = -2.5 * np.log10(zout['restU'] / zout['restV'])
vj = -2.5 * np.log10(zout['restV'] / zout['restJ'])
ssfr = zout['sfr'] / zout['mass']

sel = (zout['z_phot'] > 0.2) & (zout['z_phot'] < 1)
plt.scatter(vj[sel], uv[sel], c=np.log10(ssfr)[sel],
            vmin=-13, vmax=-8, alpha=0.5, cmap='RdYlBu')
plt.xlim(-0.2, 2.3); plt.ylim(0, 2.4); plt.grid()
plt.xlabel(r'$(V-J)_0$'); plt.ylabel(r'$(U-V)_0$')

In [ ]:
# ── Interactive visualisation ───────────────────────
import eazy.visualization
from importlib import reload
reload(eazy.visualization)

zout['jh'] = -2.5 * np.log10(self.cat['FJ1'] / self.cat['FHs'])
zout['hk'] = -2.5 * np.log10(self.cat['FHs'] / self.cat['FKs'])

bband = self.flux_columns[np.nanargmin((self.lc - 4500)**2)]
zband = self.flux_columns[np.nanargmin((self.lc - 9000)**2)]
zout['Bz'] = -2.5 * np.log10(self.cat[bband] / self.cat[zband])
zout['zK'] = -2.5 * np.log10(self.cat[zband] / self.cat['FKs'])

extra_plots = {
    'JH-redshift': ('z_phot', 'jh', 'z_phot', '(J1-Hs)_obs', (0, 4), (-0.1, 1.9)),
    'HK-redshift': ('z_phot', 'hk', 'z_phot', '(Hs-Ks)_obs', (0, 4), (-0.1, 1.9)),
    'BzK': ('Bz', 'zK', f'{bband} - {zband}', f'{zband} - Ks', (-0.5, 5), (-0.5, 5)),
}

for c in ['jh', 'hk', 'zK', 'Bz']:
    if hasattr(zout[c], 'mask'):
        zout[c].fill_value = -1.

sel = (zf['Use'] > -1)

vis = eazy.visualization.EazyExplorer(self, zout, extra_plots=extra_plots, selection=sel)

app = vis.make_dash_app(
    server_mode='inline',
    app_type='dash',
    plot_height=450,
    infer_proxy=False
)

In [ ]:
# ── Export outputs ───────────────────────────────────
import eazy.hdf5

# Full fit data for SED plotting
eazy.hdf5.write_hdf5(self, h5file=self.param['MAIN_OUTPUT_FILE'] + '.h5')

# Exported catalogue for further analysis
zout.write(f'zfourge.{field}.output.fits', overwrite=True)

print('Output files written:')
print(f'  {self.param["MAIN_OUTPUT_FILE"]}.h5')
print(f'  zfourge.{field}.output.fits')

## Output catalogue columns (`zfourge.{field}.output.fits`)

### Identifiers and coordinates
| Column | Description |
|---|---|
| `id` | Object ID |
| `ra` | Right ascension (degrees) |
| `dec` | Declination (degrees) |
| `z_spec` | Spectroscopic redshift where available; -99 or masked where not |
| `nusefilt` | Number of filters with valid flux measurements used in the fit |

### Photometric redshift
| Column | Description |
|---|---|
| `z_ml` | Maximum likelihood redshift, the peak of P(z) |
| `z_ml_chi2` | Chi-squared value at z_ml |
| `z_ml_risk` | Risk value at z_ml; higher means less reliable |
| `z_phot` | Best photometric redshift, chosen to minimise risk rather than maximise likelihood. Use this as your primary redshift estimate |
| `z_phot_chi2` | Chi-squared at z_phot |
| `z_phot_risk` | Risk at z_phot; low values indicate a well-constrained, reliable redshift |
| `z_min_risk` | Redshift at which risk is minimised (should equal z_phot) |
| `min_risk` | The minimum risk value achieved |
| `z_raw_chi2` | Redshift of the raw chi-squared minimum, before priors are applied |
| `raw_chi2` | Raw chi-squared value at z_raw_chi2 |
| `lc_min` | Shortest observed wavelength with valid flux (Angstrom) |
| `lc_max` | Longest observed wavelength with valid flux (Angstrom) |

### Redshift PDF percentiles
These describe the shape of P(z) and quantify redshift uncertainty. Wide separations between percentiles indicate a poorly constrained redshift.
| Column | Description |
|---|---|
| `z025` | 2.5th percentile of P(z) |
| `z160` | 16th percentile of P(z), lower 1-sigma bound |
| `z500` | 50th percentile of P(z), median redshift |
| `z840` | 84th percentile of P(z), upper 1-sigma bound |
| `z975` | 97.5th percentile of P(z) |

### Rest-frame fluxes
Synthesised rest-frame fluxes derived from the best-fit SED at z_phot. Used to compute rest-frame colours such as U-V and V-J for galaxy classification.
| Column | Description |
|---|---|
| `restU` | Rest-frame U-band flux |
| `restU_err` | Rest-frame U-band flux uncertainty |
| `restB` | Rest-frame B-band flux |
| `restB_err` | Rest-frame B-band flux uncertainty |
| `restV` | Rest-frame V-band flux |
| `restV_err` | Rest-frame V-band flux uncertainty |
| `restJ` | Rest-frame J-band flux |
| `restJ_err` | Rest-frame J-band flux uncertainty |

### Stellar population properties
Derived from the best-fit linear combination of spline SED templates at z_phot. These are fast estimates suitable for population-level studies; for precise individual measurements a dedicated SED fitter (e.g. CIGALE, FAST) should be used.
| Column | Description |
|---|---|
| `dL` | Luminosity distance (Mpc) |
| `mass` | Stellar mass at z_phot (solar masses) |
| `sfr` | Star formation rate averaged over the last 100 Myr at z_phot (solar masses/yr) |
| `Lv` | Rest-frame V-band luminosity (solar luminosities) |
| `LIR` | Infrared luminosity, energy absorbed and re-radiated by dust (solar luminosities) |
| `energy_abs` | Total energy absorbed by dust (solar luminosities) |
| `MLv` | Mass-to-light ratio in V-band (solar masses/solar luminosity) |
| `Av` | Dust attenuation in V-band (magnitudes); higher values mean more dust |

### Posterior percentiles (5-element arrays)
These columns contain arrays of 5 values corresponding to the [2.5, 16, 50, 84, 97.5] percentiles of each parameter's posterior distribution, computed by marginalising over the full P(z). This captures how parameter uncertainties propagate from redshift uncertainty.

For robust estimates use the median (`_p[2]`) rather than the point-estimate column. The 1-sigma uncertainty is `(_p[3] - _p[1]) / 2`.

| Column | Description |
|---|---|
| `mass_p` | Stellar mass posterior percentiles (solar masses) |
| `sfr_p` | SFR posterior percentiles (solar masses/yr) |
| `Lv_p` | V-band luminosity posterior percentiles (solar luminosities) |
| `LIR_p` | IR luminosity posterior percentiles (solar luminosities) |
| `energy_abs_p` | Absorbed energy posterior percentiles (solar luminosities) |
| `Av_p` | Dust attenuation posterior percentiles (magnitudes) |
| `ssfr_p` | Specific SFR posterior percentiles (1/yr); SFR divided by stellar mass, a measure of how actively a galaxy is forming stars relative to its existing mass |